# Phase 4: Deployment to Hugging Face Spaces

This notebook documents the deployment of the Customer Churn API to Hugging Face Spaces using Docker.

**Live API:** https://rishiteks-customer-churn-api.hf.space

---

## Overview

Hugging Face Spaces supports Docker-based deployments, which means we can run any FastAPI app with full control over the environment. The Space auto-rebuilds and redeploys every time you push to its git repository.

Two constraints HF Spaces enforces:
- The app **must listen on port 7860**
- The container **must run as a non-root user** (uid 1000)

## Step 1: Create the HF Space

1. Go to [huggingface.co](https://huggingface.co) and log in
2. Click **New Space**
3. Set the SDK to **Docker**
4. Give it a name (e.g. `customer-churn-api`)
5. Set visibility to **Public** (required for a free portfolio deployment)
6. Click **Create Space**

HF will provision a git repository for the Space. Clone it:

```bash
git clone https://huggingface.co/spaces/<your-username>/customer-churn-api
cd customer-churn-api
```

## Step 2: Two-Repo Strategy

The GitHub source repository excludes `.pkl` files (they're in `.gitignore`). The HF Space repository is separate and **does** include them.

```
GitHub repo (source)              HF Space repo (deployment)
github.com/Riz-zy/                huggingface.co/spaces/
Customer-Churn-ML                 rishiteks/customer-churn-api
│                                 │
├── app/          ──copy──►       ├── app/
├── src/          ──copy──►       ├── src/
├── models/                       ├── models/  ← .pkl files go here
│   └── (no .pkl in git)          │   ├── best_model.pkl
└── .gitignore                    │   ├── scaler.pkl
    (excludes models/*.pkl)       │   ├── feature_columns.pkl
                                  │   └── best_threshold.pkl
                                  ├── Dockerfile
                                  └── requirements.txt
```

Copy these files/folders from your local project into the cloned HF repo:
- `app/`
- `src/`
- `models/` (including the `.pkl` files — they exist locally even if not on GitHub)

## Step 3: Production requirements.txt

The development `requirements.txt` has 136 packages (Jupyter, black, pytest, etc.). The deployed container only needs 7.

Create `requirements.txt` in the HF repo with only the runtime dependencies:

```
fastapi==0.136.1
uvicorn==0.46.0
pydantic==2.13.4
scikit-learn==1.8.0
pandas==3.0.3
numpy==2.4.4
joblib==1.5.3
```

`starlette` is bundled with `fastapi` and does not need to be listed separately.

## Step 4: Dockerfile

Create a `Dockerfile` in the HF repo root:

```dockerfile
FROM python:3.11-slim

RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

WORKDIR /app

COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

COPY --chown=user . /app

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "7860"]
```

**Why each line matters:**

| Line | Reason |
|---|---|
| `python:3.11-slim` | scikit-learn 1.8.0 requires Python ≥3.11; slim keeps image small |
| `useradd -m -u 1000 user` | HF Spaces enforces non-root execution |
| `WORKDIR /app` | Relative paths like `models/best_model.pkl` resolve from here |
| `COPY requirements.txt` first | Docker layer caching — pip only re-runs when requirements change |
| `--port 7860` | HF Spaces hard requirement |
| `app.main:app` | Module `app.main` (= `app/main.py`), attribute `app` (FastAPI instance) |

## Step 5: .gitignore for HF repo

The HF repo needs a minimal `.gitignore` that does **not** exclude `models/*.pkl`:

```
__pycache__/
*.py[cod]
.env
```

That's it. The `.pkl` files must be committed here.

## Step 6: Push and Deploy

```bash
git add .
git commit -m "feat: initial deployment"
git push
```

HF Spaces picks up the push automatically and starts a Docker build. Watch progress in the **Logs** tab of your Space.

Build typically takes 2–5 minutes. When complete the Space status changes from **Building** to **Running**.

## Step 7: Test the Live API

In [ ]:
import requests

BASE_URL = "https://rishiteks-customer-churn-api.hf.space"

# Health check
response = requests.get(f"{BASE_URL}/health")
print("Health:", response.json())

In [ ]:
# Sample prediction — high-risk customer profile
customer = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 70.35,
    "TotalCharges": 140.70
}

response = requests.post(f"{BASE_URL}/predict", json=customer)
result = response.json()
print(f"Churn probability: {result['churn_probability']:.1%}")
print(f"Prediction: {'CHURN' if result['prediction'] == 1 else 'RETAIN'}")

In [ ]:
# Sample prediction — low-risk customer profile
customer_low_risk = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 60,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 55.00,
    "TotalCharges": 3300.00
}

response = requests.post(f"{BASE_URL}/predict", json=customer_low_risk)
result = response.json()
print(f"Churn probability: {result['churn_probability']:.1%}")
print(f"Prediction: {'CHURN' if result['prediction'] == 1 else 'RETAIN'}")

## Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `scikit-learn==X.X.X` not found | Wrong Python version in base image | Use `python:3.11-slim` |
| `Attribute "main" not found in module "app"` | Wrong uvicorn target | Use `app.main:app`, not `app:main` |
| `Error loading ASGI app` | Module path or attribute name wrong | Format is `<module>:<attribute>` — module uses dots, not slashes |
| `FileNotFoundError: models/best_model.pkl` | `.pkl` files not committed to HF repo | Commit the files; check HF repo `.gitignore` doesn't exclude them |
| Build succeeds but app crashes on startup | Relative path can't resolve | Ensure `WORKDIR` matches where `models/` lives |

---

## Summary

- **Space URL:** https://huggingface.co/spaces/rishiteks/customer-churn-api
- **API base URL:** https://rishiteks-customer-churn-api.hf.space
- **Docs:** https://rishiteks-customer-churn-api.hf.space/docs
- **Redeploy:** `git push` to the HF Space repo — build runs automatically